In [2]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 44.2/112.6 GB disk)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import zipfile, os
zip_path = '/content/drive/MyDrive/Damage_Detection/merged_dataset.zip'
extract_path = '/content/dataset'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("✅ Extracted!")
print(os.listdir(extract_path))

✅ Extracted!
['merged_dataset']


In [5]:
import os
label_dir = '/content/dataset/merged_dataset/valid/labels'
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
prefixes = {}
for f in os.listdir(label_dir):
    prefix = f.split('_')[0]
    prefixes[prefix] = prefixes.get(prefix, 0) + 1
    with open(f"{label_dir}/{f}", 'r') as file:
        for line in file:
            cls = int(line.split()[0])
            if cls in class_counts:
                class_counts[cls] += 1

print("Prefixes:", prefixes)
print("Class 0 (crack):", class_counts[0])
print("Class 1 (good):", class_counts[1])
print("Class 2 (scratch):", class_counts[2])
print("Class 3 (stain):", class_counts[3])

Prefixes: {'crack': 40, 'scratch': 40, 'stain': 61}
Class 0 (crack): 350
Class 1 (good): 0
Class 2 (scratch): 0
Class 3 (stain): 0


In [6]:
import os

def fix_class_ids(label_dir, prefix_to_class):
    fixed = 0
    for f in os.listdir(label_dir):
        filepath = f"{label_dir}/{f}"
        correct_class = None
        for prefix, class_id in prefix_to_class.items():
            if f.startswith(prefix):
                correct_class = class_id
                break
        if correct_class is None:
            continue
        with open(filepath, 'r') as file:
            lines = file.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                parts[0] = str(correct_class)
                new_lines.append(' '.join(parts) + '\n')
        with open(filepath, 'w') as file:
            file.writelines(new_lines)
        fixed += 1
    print(f"Fixed {fixed} files in {label_dir}")

prefix_to_class = {
    'crack':   0,
    'good':    1,
    'scratch': 2,
    'stain':   3
}

for split in ['train', 'valid', 'test']:
    label_dir = f'/content/dataset/merged_dataset/{split}/labels'
    fix_class_ids(label_dir, prefix_to_class)

print("\n✅ All class IDs fixed!")

Fixed 1001 files in /content/dataset/merged_dataset/train/labels
Fixed 141 files in /content/dataset/merged_dataset/valid/labels
Fixed 106 files in /content/dataset/merged_dataset/test/labels

✅ All class IDs fixed!


In [7]:
import os

label_dir = '/content/dataset/merged_dataset/valid/labels'
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}

for f in os.listdir(label_dir):
    with open(f"{label_dir}/{f}", 'r') as file:
        for line in file:
            cls = int(line.split()[0])
            if cls in class_counts:
                class_counts[cls] += 1

print("Class 0 (crack):", class_counts[0])
print("Class 1 (good):", class_counts[1])
print("Class 2 (scratch):", class_counts[2])
print("Class 3 (stain):", class_counts[3])

Class 0 (crack): 40
Class 1 (good): 0
Class 2 (scratch): 142
Class 3 (stain): 168


In [8]:
import yaml

yaml_path = '/content/dataset/merged_dataset/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = '/content/dataset/merged_dataset/train/images'
data['val']   = '/content/dataset/merged_dataset/valid/images'
data['test']  = '/content/dataset/merged_dataset/test/images'
data['nc']    = 4
data['names'] = ['crack', 'good', 'scratch', 'stain']

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml updated!")
print(data)

✅ data.yaml updated!
{'train': '/content/dataset/merged_dataset/train/images', 'val': '/content/dataset/merged_dataset/valid/images', 'test': '/content/dataset/merged_dataset/test/images', 'nc': 4, 'names': ['crack', 'good', 'scratch', 'stain']}


In [9]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/dataset/merged_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='phone_damage_v2',
    patience=10,
    save=True,
    plots=True,
    device=0
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=phone_damage_v24, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b47d51608c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [10]:
from ultralytics import YOLO

model = YOLO('/content/runs/detect/phone_damage_v2/weights/best.pt')
results = model.val(data='/content/dataset/merged_dataset/data.yaml')

print("\n" + "="*60)
print("       TrustLensAI - PHONE DAMAGE DETECTION MODEL v2")
print("="*60)

print("\n📊 OVERALL METRICS:")
print(f"  Precision:     {results.results_dict['metrics/precision(B)']:.4f}  ({results.results_dict['metrics/precision(B)']*100:.2f}%)")
print(f"  Recall:        {results.results_dict['metrics/recall(B)']:.4f}  ({results.results_dict['metrics/recall(B)']*100:.2f}%)")
print(f"  mAP50:         {results.results_dict['metrics/mAP50(B)']:.4f}  ({results.results_dict['metrics/mAP50(B)']*100:.2f}%)")
print(f"  mAP50-95:      {results.results_dict['metrics/mAP50-95(B)']:.4f}  ({results.results_dict['metrics/mAP50-95(B)']*100:.2f}%)")

print("\n📦 PER CLASS METRICS:")
print(f"  {'Class':<12} {'mAP50':>10} {'mAP50-95':>12} {'Status':>10}")
print(f"  {'-'*46}")
class_names = ['crack', 'scratch', 'stain']
maps = results.maps
for i, (name, m50, m5095) in enumerate(zip(class_names, [maps[0], maps[2], maps[3]], [maps[0], maps[2], maps[3]])):
    status = "🔥 Excellent" if maps[i] > 0.95 else "✅ Good" if maps[i] > 0.85 else "⚠️ Needs Work"
    print(f"  {name:<12} {maps[i]:>10.4f} {'N/A':>12} {status:>10}")

print("\n⚡ SPEED METRICS:")
print(f"  Preprocess:    {results.speed['preprocess']:.2f} ms/image")
print(f"  Inference:     {results.speed['inference']:.2f} ms/image")
print(f"  Postprocess:   {results.speed['postprocess']:.2f} ms/image")
print(f"  Total:         {results.speed['preprocess'] + results.speed['inference'] + results.speed['postprocess']:.2f} ms/image")

print("\n🎯 MODEL SUMMARY:")
print(f"  Model:         YOLOv8n (nano)")
print(f"  Parameters:    3,006,428")
print(f"  Classes:       crack, scratch, stain")
print(f"  Image Size:    640x640")
print(f"  Training Time: 0.387 hours (~23 minutes)")
print(f"  Best Epoch:    38 (mAP50-95: 0.731)")

print("\n✅ MODEL STATUS: PRODUCTION READY")
print("="*60)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2229.5±843.2 MB/s, size: 291.7 KB)
val: Scanning /content/dataset/merged_dataset/valid/labels.cache... 141 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 141/141 59.1Mit/s 0.0s
val: /content/dataset/merged_dataset/valid/images/stain_Sta_0024_jpg.rf.830d8e9482c0a471e8d209bf3a2f0bf4.jpg: 1 duplicate labels removed
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 207, len(boxes) = 349. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.6it/s 5.7s
                   all        141        349      0.981      0.978     

In [11]:
import shutil
shutil.copy(
    '/content/runs/detect/phone_damage_v2/weights/best.pt',
    '/content/drive/MyDrive/best_v2.pt'
)
print("✅ best_v2.pt saved to Drive!")

✅ best_v2.pt saved to Drive!


In [12]:
def calculate_severity(image_path):
    results = model(image_path, verbose=False)[0]

    img_h, img_w = results.orig_shape
    img_area = img_h * img_w

    if len(results.boxes) == 0:
        return {
            'severity_score': 0,
            'condition': 'Good',
            'detections': [],
            'explanation': 'No damage detected. Phone appears to be in good condition.'
        }

    detections = []
    total_weighted_score = 0

    for box in results.boxes:
        class_id = int(box.cls)
        class_name = model.names[class_id]
        confidence = float(box.conf)

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        box_area = (x2 - x1) * (y2 - y1)
        area_pct = (box_area / img_area) * 100

        weight = CLASS_WEIGHTS.get(class_name, 0.5)
        detection_score = area_pct * weight

        detections.append({
            'class': class_name,
            'confidence': round(confidence * 100, 1),
            'area_pct': round(area_pct, 2),
            'score_contribution': round(detection_score, 3)
        })

        total_weighted_score += detection_score

    # Better normalization:
    # 100% area × 1.0 weight = 100 → should be 10/10
    # So divide by 10
    raw_score = total_weighted_score / 10

    # Apply square root to make scoring less aggressive for large areas
    import math
    severity_score = min(round(math.sqrt(raw_score) * 3.16, 1), 10)
    severity_score = max(severity_score, 1)

    if severity_score <= 2:
        condition = 'Minor Damage'
    elif severity_score <= 4:
        condition = 'Moderate Damage'
    elif severity_score <= 7:
        condition = 'Severe Damage'
    else:
        condition = 'Critical Damage'

    return {
        'severity_score': severity_score,
        'condition': condition,
        'detections': detections,
        'total_weighted_score': round(total_weighted_score, 3)
    }

print("✅ Severity scoring updated!")

✅ Severity scoring updated!


In [13]:
# Test with a sample image from your dataset
import os

CLASS_WEIGHTS = {'crack': 1.0, 'scratch': 0.6, 'stain': 0.4}

test_image = '/content/dataset/merged_dataset/test/images'
sample = os.listdir(test_image)[0]

result = calculate_severity(f"{test_image}/{sample}")

print(f"\n📱 Image: {sample}")
print(f"🔢 Severity Score: {result['severity_score']} / 10")
print(f"📊 Condition: {result['condition']}")
print(f"\n🔍 Detections:")
for d in result['detections']:
    print(f"  → {d['class']} | Confidence: {d['confidence']}% | Area: {d['area_pct']}% | Score contribution: {d['score_contribution']}")


📱 Image: crack_203k_jpeg.rf.c4a833bcbfa9eac3f9e70047e26663af.jpg
🔢 Severity Score: 7.6 / 10
📊 Condition: Critical Damage

🔍 Detections:
  → crack | Confidence: 94.7% | Area: 57.71% | Score contribution: 57.706


In [14]:
import os
import random

test_dir = '/content/dataset/merged_dataset/test/images'
all_images = os.listdir(test_dir)

# Pick 2 of each class
crack_imgs   = [f for f in all_images if f.startswith('crack')][:2]
scratch_imgs = [f for f in all_images if f.startswith('scratch')][:2]
stain_imgs   = [f for f in all_images if f.startswith('stain')][:2]

test_samples = crack_imgs + scratch_imgs + stain_imgs

print("="*60)
print("     SEVERITY SCORING - MULTI CLASS TEST")
print("="*60)

for img_name in test_samples:
    img_path = f"{test_dir}/{img_name}"
    result = calculate_severity(img_path)

    print(f"\n📱 {img_name[:40]}")
    print(f"   Score     : {result['severity_score']} / 10")
    print(f"   Condition : {result['condition']}")
    for d in result['detections']:
        print(f"   → {d['class']:<10} | Conf: {d['confidence']}% | Area: {d['area_pct']}%")
    print(f"   {'-'*45}")

print("\n✅ Multi-class severity test complete!")

     SEVERITY SCORING - MULTI CLASS TEST

📱 crack_203k_jpeg.rf.c4a833bcbfa9eac3f9e70
   Score     : 7.6 / 10
   Condition : Critical Damage
   → crack      | Conf: 94.7% | Area: 57.71%
   ---------------------------------------------

📱 crack_279k_jpeg.rf.ca4b5a918e60f54005172
   Score     : 8.9 / 10
   Condition : Critical Damage
   → crack      | Conf: 93.1% | Area: 79.02%
   ---------------------------------------------

📱 scratch_Scr_0167_jpg.rf.5bd4fd7b24ec31fe
   Score     : 2.1 / 10
   Condition : Moderate Damage
   → scratch    | Conf: 95.1% | Area: 3.15%
   → scratch    | Conf: 87.1% | Area: 1.5%
   → scratch    | Conf: 87.0% | Area: 1.79%
   → scratch    | Conf: 84.4% | Area: 0.88%
   → scratch    | Conf: 67.4% | Area: 0.13%
   ---------------------------------------------

📱 scratch_Scr_0315_jpg.rf.56c1d38f1a5640e3
   Score     : 1.1 / 10
   Condition : Minor Damage
   → scratch    | Conf: 92.2% | Area: 1.44%
   → scratch    | Conf: 82.6% | Area: 0.47%
   → scratch    | Conf

In [15]:
def generate_explanation(result):
    score = result['severity_score']
    condition = result['condition']
    detections = result['detections']

    if score == 0:
        return "✅ No damage detected. Phone appears to be in excellent condition and is suitable for resale."

    # Group detections by class
    class_summary = {}
    for d in detections:
        cls = d['class']
        if cls not in class_summary:
            class_summary[cls] = {'count': 0, 'max_area': 0, 'total_area': 0}
        class_summary[cls]['count'] += 1
        class_summary[cls]['max_area'] = max(class_summary[cls]['max_area'], d['area_pct'])
        class_summary[cls]['total_area'] += d['area_pct']

    # Build damage description
    damage_parts = []

    if 'crack' in class_summary:
        c = class_summary['crack']
        count = c['count']
        area = round(c['total_area'], 1)
        if area > 50:
            severity_word = "severe"
        elif area > 20:
            severity_word = "significant"
        else:
            severity_word = "minor"
        crack_text = f"{count} {severity_word} crack{'s' if count > 1 else ''} covering {area}% of the surface"
        damage_parts.append(crack_text)

    if 'scratch' in class_summary:
        s = class_summary['scratch']
        count = s['count']
        area = round(s['total_area'], 1)
        if count == 1:
            scratch_text = f"1 scratch covering {area}% of the surface"
        else:
            scratch_text = f"{count} scratches covering {area}% of the surface"
        damage_parts.append(scratch_text)

    if 'stain' in class_summary:
        st = class_summary['stain']
        count = st['count']
        area = round(st['total_area'], 2)
        if count == 1:
            stain_text = f"1 stain detected ({area}% area)"
        else:
            stain_text = f"{count} stains detected ({area}% total area)"
        damage_parts.append(stain_text)

    # Join damage parts
    if len(damage_parts) == 1:
        damage_description = damage_parts[0]
    elif len(damage_parts) == 2:
        damage_description = f"{damage_parts[0]} and {damage_parts[1]}"
    else:
        damage_description = ", ".join(damage_parts[:-1]) + f", and {damage_parts[-1]}"

    # Recommendation based on score
    if score <= 2:
        recommendation = "Suitable for resale with minor cosmetic disclosure."
    elif score <= 4:
        recommendation = "Suitable for resale at reduced price with damage disclosure."
    elif score <= 7.5:
        recommendation = "Requires repair before resale. Price should reflect damage."
    else:
        recommendation = "Not suitable for resale without significant repair."

    # Final explanation
    explanation = (
        f"Damage detected on this device: {damage_description}. "
        f"Severity Score: {score}/10 ({condition}). "
        f"Recommendation: {recommendation}"
    )

    return explanation


print("✅ Explanation generator ready!")

# Test it
print("\n" + "="*60)
print("     EXPLANATION GENERATOR TEST")
print("="*60)

for img_name in test_samples:
    img_path = f"{test_dir}/{img_name}"
    result = calculate_severity(img_path)
    explanation = generate_explanation(result)
    print(f"\n📱 {img_name[:45]}")
    print(f"   Score : {result['severity_score']}/10 | {result['condition']}")
    print(f"   💬 {explanation}")
    print()

✅ Explanation generator ready!

     EXPLANATION GENERATOR TEST

📱 crack_203k_jpeg.rf.c4a833bcbfa9eac3f9e70047e2
   Score : 7.6/10 | Critical Damage
   💬 Damage detected on this device: 1 severe crack covering 57.7% of the surface. Severity Score: 7.6/10 (Critical Damage). Recommendation: Not suitable for resale without significant repair.


📱 crack_279k_jpeg.rf.ca4b5a918e60f54005172ba437
   Score : 8.9/10 | Critical Damage
   💬 Damage detected on this device: 1 severe crack covering 79.0% of the surface. Severity Score: 8.9/10 (Critical Damage). Recommendation: Not suitable for resale without significant repair.


📱 scratch_Scr_0167_jpg.rf.5bd4fd7b24ec31febe456
   Score : 2.1/10 | Moderate Damage
   💬 Damage detected on this device: 5 scratches covering 7.5% of the surface. Severity Score: 2.1/10 (Moderate Damage). Recommendation: Suitable for resale at reduced price with damage disclosure.


📱 scratch_Scr_0315_jpg.rf.56c1d38f1a5640e3349ab
   Score : 1.1/10 | Minor Damage
   💬 Damage 

In [16]:
!pip install fastapi uvicorn python-multipart nest-asyncio pyngrok
print("✅ FastAPI installed!")

✅ FastAPI installed!


In [17]:
import nest_asyncio
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import uvicorn
import shutil, os, math
from ultralytics import YOLO
from google.colab.output import eval_js
import threading

nest_asyncio.apply()

# Load model
model = YOLO('/content/drive/MyDrive/best_v2.pt')

CLASS_WEIGHTS = {'crack': 1.0, 'scratch': 0.6, 'stain': 0.4}

app = FastAPI(title="TrustLensAI - Phone Damage Detection API")

def calculate_severity(image_path):
    results = model(image_path, verbose=False)[0]
    img_h, img_w = results.orig_shape
    img_area = img_h * img_w

    if len(results.boxes) == 0:
        return {'severity_score': 0, 'condition': 'Good', 'detections': []}

    detections = []
    total_weighted_score = 0

    for box in results.boxes:
        class_id = int(box.cls)
        class_name = model.names[class_id]
        confidence = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        box_area = (x2 - x1) * (y2 - y1)
        area_pct = (box_area / img_area) * 100
        weight = CLASS_WEIGHTS.get(class_name, 0.5)
        detection_score = area_pct * weight
        detections.append({
            'class': class_name,
            'confidence': round(confidence * 100, 1),
            'area_pct': round(area_pct, 2),
            'score_contribution': round(detection_score, 3)
        })
        total_weighted_score += detection_score

    raw_score = total_weighted_score / 10
    severity_score = min(round(math.sqrt(raw_score) * 3.16, 1), 10)
    severity_score = max(severity_score, 1)

    if severity_score <= 2:
        condition = 'Minor Damage'
    elif severity_score <= 4:
        condition = 'Moderate Damage'
    elif severity_score <= 7.5:
        condition = 'Severe Damage'
    else:
        condition = 'Critical Damage'

    return {'severity_score': severity_score, 'condition': condition, 'detections': detections}

def generate_explanation(result):
    score = result['severity_score']
    condition = result['condition']
    detections = result['detections']

    if score == 0:
        return "No damage detected. Phone appears to be in excellent condition and is suitable for resale."

    class_summary = {}
    for d in detections:
        cls = d['class']
        if cls not in class_summary:
            class_summary[cls] = {'count': 0, 'total_area': 0}
        class_summary[cls]['count'] += 1
        class_summary[cls]['total_area'] += d['area_pct']

    damage_parts = []
    if 'crack' in class_summary:
        c = class_summary['crack']
        area = round(c['total_area'], 1)
        word = "severe" if area > 50 else "significant" if area > 20 else "minor"
        damage_parts.append(f"{c['count']} {word} crack{'s' if c['count']>1 else ''} covering {area}% of the surface")
    if 'scratch' in class_summary:
        s = class_summary['scratch']
        damage_parts.append(f"{s['count']} scratch{'es' if s['count']>1 else ''} covering {round(s['total_area'],1)}% of the surface")
    if 'stain' in class_summary:
        st = class_summary['stain']
        damage_parts.append(f"{st['count']} stain{'s' if st['count']>1 else ''} detected ({round(st['total_area'],2)}% total area)")

    if len(damage_parts) == 1:
        damage_description = damage_parts[0]
    elif len(damage_parts) == 2:
        damage_description = f"{damage_parts[0]} and {damage_parts[1]}"
    else:
        damage_description = ", ".join(damage_parts[:-1]) + f", and {damage_parts[-1]}"

    if score <= 2:
        recommendation = "Suitable for resale with minor cosmetic disclosure."
    elif score <= 4:
        recommendation = "Suitable for resale at reduced price with damage disclosure."
    elif score <= 7.5:
        recommendation = "Requires repair before resale. Price should reflect damage."
    else:
        recommendation = "Not suitable for resale without significant repair."

    return f"Damage detected: {damage_description}. Severity Score: {score}/10 ({condition}). Recommendation: {recommendation}"


@app.get("/")
def root():
    return {"message": "TrustLensAI Phone Damage Detection API", "status": "running", "version": "2.0"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    temp_path = f"/content/temp_{file.filename}"
    with open(temp_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    try:
        result = calculate_severity(temp_path)
        explanation = generate_explanation(result)
        os.remove(temp_path)
        return JSONResponse(content={
            "status": "success",
            "severity_score": result['severity_score'],
            "condition": result['condition'],
            "explanation": explanation,
            "detections": result['detections']
        })
    except Exception as e:
        os.remove(temp_path)
        return JSONResponse(status_code=500, content={"status": "error", "message": str(e)})

@app.get("/health")
def health():
    return {"status": "healthy", "model": "phone_damage_v2", "classes": ["crack", "scratch", "stain"]}


# Start server in background thread
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

import time
time.sleep(3)

# Get Colab public URL
colab_url = eval_js("google.colab.kernel.proxyPort(8000)")
print(f"\n✅ API is live at: {colab_url}")
print(f"📖 Docs available at: {colab_url}/docs")
print(f"\n🧪 Test with:")
print(f'curl -X POST "{colab_url}/predict" -F "file=@your_image.jpg"')

INFO:     Started server process [27759]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



✅ API is live at: https://8000-gpu-t4-s-26dw89sj32zt4-a.us-west4-0.prod.colab.dev
📖 Docs available at: https://8000-gpu-t4-s-26dw89sj32zt4-a.us-west4-0.prod.colab.dev/docs

🧪 Test with:
curl -X POST "https://8000-gpu-t4-s-26dw89sj32zt4-a.us-west4-0.prod.colab.dev/predict" -F "file=@your_image.jpg"
